# Subject Dependent Experiments
model trained on one individual subject's samples, shuffled and divided into an 80-20 train-test split

In [1]:
import numpy as np
import tensorflow as tf
from data_preprocess import extract_subj_dep
import hybrid_model_structures as models

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from keras.optimizers import Adam
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, cohen_kappa_score, confusion_matrix
import os
# import pandas as pd
# import openpyxl

# data root
dir_path_f = "../../Data/EF-WG/Raw/"
dir_path_e = "../../Data/EF-WG/Raw/"

# result logging
if not os.path.exists('Results'):
    os.makedirs('Results')
result_file="Results/Log.txt"
log_excel = 'Results/Log.xlsx'

# hyperparameters
baseline_model=False # EF-Net or ResNet (basleine)
resnet=False
if baseline_model:
    lr=0.00005
    EPOCHS=50
else:
    lr=0.001
    EPOCHS=70
BATCH_SIZE = 32
RANDOM_STATE=38
n_subject = 26

acc_list, precision_list, recall_list, f1_list, kappa_list = [], [], [], [], []
for subject in range(n_subject):
    subject += 1
    subject = f'{subject:02d}'
    X_eeg, X_fnirs, Y = extract_subj_dep(subject,baseline_model)

    # Standardize our signals data
    # Reshaping the data samples into 2D applying the scaling and then reshaping back
    scaler = StandardScaler()
    XeegS = scaler.fit_transform(X_eeg.reshape(-1, X_eeg.shape[-1])).reshape(X_eeg.shape)
    XfnirsS = scaler.fit_transform(X_fnirs.reshape(-1, X_fnirs.shape[-1])).reshape(X_fnirs.shape)

    # adjustment to labels to make compatible with binary_crossentropy
    Y=Y[:,0]

    print(XeegS.shape)
    print(XfnirsS.shape)
    print(Y.shape)

    # reshape data
    if baseline_model:
        XeegS_3D=XeegS
        XfnirsS_3D=XfnirsS
    else:
        # introduce an extra dimension
        XeegS_3D=np.reshape(XeegS,(XeegS.shape[0],XeegS.shape[1],XeegS.shape[2],1))
        XfnirsS_3D=np.reshape(XfnirsS,(XfnirsS.shape[0],XfnirsS.shape[1],XfnirsS.shape[2],1))
    XeegS_3D.shape

    # split data into training and testing sets (60/40 ratio)
    Xeeg_train, Xeeg_test, y_train, y_test = train_test_split(XeegS_3D, Y, test_size=0.4,random_state=RANDOM_STATE)
    print(Xeeg_train.shape,Xeeg_test.shape)

    Xfnirs_train, Xfnirs_test, y_train2, y_test2 = train_test_split(XfnirsS_3D, Y, test_size=0.4, random_state=RANDOM_STATE)
    print(Xfnirs_train.shape, Xfnirs_test.shape)

    if baseline_model:
        hybridmodel= models.eeg_fnirs_vgg(Xeeg_train,Xfnirs_train, resnet)
    else:
        hybridmodel=models.eeg_fnirs_cnn_v2()
    hybridmodel.summary()
    hybridmodel.compile(optimizer=Adam(learning_rate=lr),loss='binary_crossentropy',metrics=['accuracy'])

    # save weights from the highest accuracy epoch
    checkpoint_filepath = '/tmp/checkpoint.weights.h5'
    model_checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
        filepath=checkpoint_filepath,
        save_weights_only=True,
        monitor='val_accuracy',
        mode='max',
        save_best_only=True,
        verbose=1)

    hybridmodel.fit(x=[Xeeg_train, Xfnirs_train], y=y_train, validation_data=([Xeeg_test, Xfnirs_test], y_test), epochs=EPOCHS, callbacks=[model_checkpoint_callback])
    
    hybridmodel.load_weights(checkpoint_filepath)
    probabilities = hybridmodel.predict([Xeeg_test, Xfnirs_test])
    y_preds= np.round(probabilities)
    loss,accuracy = hybridmodel.evaluate([Xeeg_test,Xfnirs_test], y_test)

    accuracy = accuracy_score(y_test, y_preds)
    precision = precision_score(y_test, y_preds)
    recall = recall_score(y_test, y_preds)
    f1 = f1_score(y_test, y_preds)
    kappa = cohen_kappa_score(y_test, y_preds)

    # acc_list.append(accuracy)
    # precision_list.append(precision)
    # recall_list.append(recall)
    # f1_list.append(f1)
    # kappa_list.append(kappa)

    log_path = 'Results/Subject' + subject + '.txt'
    with open(log_path, 'a') as log_file:
        log_file.write(f'Accuracy: {accuracy:.2f}\n')
        log_file.write(f'Precision: {precision:.2f}\n')
        log_file.write(f'Recall: {recall:.2f}\n')
        log_file.write(f'F1: {f1:.2f}\n')
        log_file.write(f'Kappa: {kappa:.2f}\n')
        log_file.write('\n')

    # new_row = pd.DataFrame([[accuracy, precision, recall, f1, kappa]], 
    #                     columns=['Accuracy', 'Std_Accuracy', 'Precision', 'Std_Precision', 'Recall', 'Std_Recall', 'F1', 'Std_F1', 'Kappa'])
    # new_row = new_row.round(2)
    # if not os.path.exists(log_excel):
    #     new_row.to_excel(log_excel, index=False, engine='openpyxl')
    # else:
    #     with pd.ExcelWriter(log_excel, mode='a', engine='openpyxl', if_sheet_exists='overlay') as writer:
    #         if 'Sheet1' in writer.sheets:
    #             startrow = writer.sheets['Sheet1'].max_row
    #         else:
    #             startrow = 0
    #         new_row.to_excel(writer, index=False, header=False, startrow=startrow)

2025-05-08 15:11:57.223727: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-05-08 15:11:57.241279: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-05-08 15:11:57.246630: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-05-08 15:11:57.260691: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


tasks shape (100, 72)
Length of data samples,  6
tasks shape (100, 72)
Length of data samples,  12
tasks shape (100, 72)
Length of data samples,  18
tasks shape (100, 72)
Length of data samples,  24
tasks shape (100, 72)
Length of data samples,  30
tasks shape (100, 72)
Length of data samples,  36
tasks shape (100, 72)
Length of data samples,  42
tasks shape (100, 72)
Length of data samples,  48
tasks shape (100, 72)
Length of data samples,  54
tasks shape (100, 72)
Length of data samples,  60
tasks shape (100, 72)
Length of data samples,  66
tasks shape (100, 72)
Length of data samples,  72
tasks shape (100, 72)
Length of data samples,  78
tasks shape (100, 72)
Length of data samples,  84
tasks shape (100, 72)
Length of data samples,  90
tasks shape (100, 72)
Length of data samples,  96
tasks shape (100, 72)
Length of data samples,  102
tasks shape (100, 72)
Length of data samples,  108
tasks shape (100, 72)
Length of data samples,  114
tasks shape (100, 72)
Length of data samples,  1

2025-05-08 15:12:00.360625: I tensorflow/core/common_runtime/gpu/gpu_device.cc:2021] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 56679 MB memory:  -> device: 0, name: NVIDIA A800 80GB PCIe, pci bus id: 0000:a1:00.0, compute capability: 8.0


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 500, 30,   │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 494, 30,   │        256 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 488, 30,   │      7,200 │ conv2d[0][0]      │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 482, 30,   │      7,200 │ conv2d_1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 68, 30,    │          0 │ conv2d_2[0][0]    │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, 25, 72, 1) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 68, 30,    │          0 │ max_pooling2d[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_6 (Conv2D)   │ (None, 22, 72,    │        160 │ input_layer_1[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 68, 30,    │        128 │ dropout[0][0]     │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_7 (Conv2D)   │ (None, 19, 72,    │      4,128 │ conv2d_6[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 65, 27,    │     32,832 │ batch_normalizat… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 4, 72, 32) │          0 │ conv2d_7[0][0]    │
│ (MaxPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 62, 24,    │     65,600 │ conv2d_3[0][0]    │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 4, 72, 32) │          0 │ max_pooling2d_2[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 59, 21,    │     65,600 │ conv2d_4[0][0]    │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 4, 72, 32) │        128 │ dropout_3[0][0]   │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 14, 5, 64) │          0 │ conv2d_5[0][0]    │
│ (MaxPooling2D)      │                   │            │                 

...

 Trainable params: 1,757,249 (6.70 MB)

 Non-trainable params: 384 (1.50 KB)

Epoch 1/70


/data_main/junlin/conda_env/mbi_tf/lib/python3.10/site-packages/keras/src/models/functional.py:225: UserWarning: The structure of `inputs` doesn't match the expected structure: ['keras_tensor', 'keras_tensor_1']. Received: the structure of inputs=('*', '*')
  warnings.warn(


7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 948ms/step - accuracy: 0.5194 - loss: 0.6935
Epoch 1: val_accuracy improved from -inf to 0.46528, saving model to /tmp/checkpoint.weights.h5
7/7 ━━━━━━━━━━━━━━━━━━━━ 17s 1s/step - accuracy: 0.5158 - loss: 0.6936 - val_accuracy: 0.4653 - val_loss: 0.6979
Epoch 2/70
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.5672 - loss: 0.6906
Epoch 2: val_accuracy improved from 0.46528 to 0.49306, saving model to /tmp/checkpoint.weights.h5
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 0.5710 - loss: 0.6898 - val_accuracy: 0.4931 - val_loss: 0.6870
Epoch 3/70
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.6521 - loss: 0.6758
Epoch 3: val_accuracy improved from 0.49306 to 0.63194, saving model to /tmp/checkpoint.weights.h5
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 0.6452 - loss: 0.6761 - val_accuracy: 0.6319 - val_loss: 0.6795
Epoch 4/70
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.5422 - loss: 0.6704
Epoch 4: val_accuracy improved from 0.63194